In [1]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE, URL = '/content/code', 'github.com/quocthangtrann/computer_vision_ADAS.git'
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
def sh(cmd):       # chạy lệnh, GOM cả stdout và stderr (stderr của lệnh con không hiện trong ô kết quả Colab nếu không gom)
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
if not os.path.exists(CODE):
    r = sh(f'git clone https://{URL} {CODE}')
    if r.returncode: raise SystemExit('git clone lỗi:\n' + (r.stderr or r.stdout))
else:
    r = sh(f'git -C {CODE} pull --ff-only')
    if r.returncode:       # thường do sửa tay tệp trong /content/code: cất vào stash (KHÔNG xóa) rồi pull lại
        print('git pull lỗi:\n' + (r.stderr or r.stdout).strip()[-800:])
        print('thay đổi cục bộ trong', CODE + ':\n' + (sh(f'git -C {CODE} status --short').stdout.strip() or '(không có)'))
        r = sh(f'git -C {CODE} stash push -u -m deploy-auto && git -C {CODE} pull --ff-only')
        if r.returncode: raise SystemExit('pull vẫn lỗi sau khi stash (không phải do sửa cục bộ):\n' + (r.stderr or r.stdout).strip()[-800:])
        print('đã cất thay đổi cục bộ vào stash (xem lại: git -C /content/code stash list) và pull xong')
subprocess.run(f'rsync -a --exclude .git {CODE}/repo/ /content/drive/MyDrive/capstone_ai/repo/', shell=True, check=True)
try: import segmentation_models_pytorch
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
os.chdir('/content/drive/MyDrive/capstone_ai')
print(subprocess.run(f'git -C {CODE} log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip())
print('mã mới nhất đã ở capstone_ai/repo/ (runtime/, config/, tests/ chạy thẳng từ /content/code) | thư mục làm việc:', os.getcwd())


Mounted at /content/drive
8e96ba9 B2 (hd640s4_p1s5_kdT): 40_int8 và 30_eval trỏ gói riêng _s4, không đè gói B; FREEZE mặc định s0
mã mới nhất đã ở capstone_ai/repo/ (runtime/, config/, tests/ chạy thẳng từ /content/code) | thư mục làm việc: /content/drive/MyDrive/capstone_ai


In [2]:
import os, subprocess, sys
os.chdir('/content/drive/MyDrive/capstone_ai'); os.makedirs('runs', exist_ok=True)
# CẤU HÌNH. Mặc định B (mô hình chính): fsnet_m_p1s5 640x320 -> handoff_s3b_fsnet_m_p1s5_640x320.
# A (dự phòng, ngừng phát triển): W, H, ARCH = 320, 160, 'fsnet_m'; RUN = 'runs/b9s4_c4aug_s0' -> handoff_s3b
W, H, ARCH = 640, 320, 'fsnet_m_p1s5'
RUN = 'runs/hd640s4_p1s5_kdT_c4aug_s0'
OUT = 'handoff_s3b' if (W, H) == (320, 160) else f'handoff_s3b_{ARCH}_{W}x{H}_s4'   # gói HD ghi ra thư mục khác, không đè gói 320x160
SZ = ['--arch', ARCH, '--w', str(W), '--h', str(H)]

def run(script, *args):
    cmd = [sys.executable, '-u', f'repo/{script}', *args]
    print('>>>', ' '.join(cmd), flush=True)
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'runs/{script[:-3]}.log', 'a') as lg:
        for ln in pr.stdout:
            lg.write(ln); print(ln, end='', flush=True)
    if pr.wait() != 0: raise SystemExit(f'{script} dừng (mã thoát {pr.returncode}): dán đoạn log trên cho tôi, KHÔNG chạy bước sau')

# Chạy được trên runtime CPU. Bước FP32 của int8_sim so ONNX với final_metrics.json của run: lệch quá 0,002 thì dừng.
if (W, H) != (320, 160): run('make_hd_calib.py', '--w', str(W), '--h', str(H))   # 200 ảnh hiệu chuẩn + 100 ảnh test HD; có rồi thì bỏ qua
run('int8_sim.py', '--run', RUN, '--out', OUT, *SZ)
run('int8_pack.py', '--run', RUN, '--out', OUT, *SZ)
print(f'XONG: gói {OUT}/ (và {OUT}_v1.zip). Bước sau: 30_eval (runtime trên ảnh test với gói này).')


>>> /usr/bin/python3 -u repo/make_hd_calib.py --w 640 --h 320
đã có 200 ảnh hiệu chuẩn ở /content/drive/MyDrive/capstone_ai/data/v2/hd_calib_640x320 và 100 ảnh test ở /content/drive/MyDrive/capstone_ai/data/v2/hd_test_640x320; dùng --force để dựng lại
>>> /usr/bin/python3 -u repo/int8_sim.py --run runs/hd640s4_p1s5_kdT_c4aug_s0 --out handoff_s3b_fsnet_m_p1s5_640x320_s4 --arch fsnet_m_p1s5 --w 640 --h 320
=== 1) xuất ONNX (fsnet_m_p1s5, 640x320)
epoch: 60 | toán tử: ['Concat', 'Constant', 'Conv', 'MaxPool', 'Relu', 'Resize']
17 conv | 0.7248 GMAC | 1.316 M tham số | lưới logit 80x40 | sha256 onnx: 4896925f3aed06c55201be885063bfd542f3f03a069d14b585f7f88106876b77
  dấu vân tay trọng số (không phụ thuộc phiên bản torch/onnx): 94b3546126e014251defab121debcba944453673ac0d5d092d7f7060217a35f7
=== 2) FP32 (ONNX) trên 408 ảnh test (có nhãn), IoU ở lưới 320x160
  ONNX FP32 : [np.float64(0.9674), np.float64(0.5298), np.float64(0.8266), np.float64(0.6781)] | IoU đường 0.8995
  final_metrics (iou_3